# Tutorial: Generative AI for chemical design

In [ ]:
! git clone --depth=1 https://github.com/PhilLecl/TutorialChemGenAI.git data

## Load modules and data

In [ ]:
! pip install pytoda &>> /dev/null
! pip install "positional-encodings[pytorch]" &>> /dev/null
! pip install pyscf &>> /dev/null

In [ ]:
import json
import math
from typing import Optional

import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
import torch
from positional_encodings.torch_encodings import PositionalEncoding1D
from pytoda.smiles import SMILESTokenizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from torch import nn
from torch.distributions import Normal, kl_divergence
from torch.nn.utils.rnn import pad_sequence
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm
from pyscf import gto, dft

In [ ]:
device = (
  torch.accelerator.current_accelerator()
  if torch.accelerator.is_available()
  else torch.device("cpu")
)

### Load dataset and split into training and validation subset

In [ ]:
# load dataframe
df = pd.read_csv("data/qm9.csv")

# keep only relevant columns
df = df[
  [
    "smiles",
    "mu",  # dipole moment / D
    "homo",  # HOMO energy / Eh
    "lumo",  # LUMO energy / Eh
    "gap",  # HOMO-LUMO gap / Eh
  ]
]

# split into training and validation subset
df_train, df_valid = train_test_split(
  df,
  test_size=0.2,
  random_state=20260528,
)

# Scale values to have zero mean and unit variance
scaler = StandardScaler().fit(df_train.iloc[:, 1:])
df_train.iloc[:, 1:] = scaler.transform(df_train.iloc[:, 1:])
df_valid.iloc[:, 1:] = scaler.transform(df_valid.iloc[:, 1:])

## Initialize tokenizer and define vocabulary

In [ ]:
augmented_tokenizer = SMILESTokenizer(
  add_start_and_stop=True, augment=True
)
# augmented_tokenizer.add_dataset(df_train["smiles"])
# augmented_tokenizer.save_vocabulary("data/vocab.json")
augmented_tokenizer.load_vocabulary("data/vocab.json")
canonical_tokenizer = SMILESTokenizer(
  add_start_and_stop=True, canonical=True
)
canonical_tokenizer.load_vocabulary("data/vocab.json")

**Show the tokens defined in the vocabulary and the associated indices**


In [ ]:
print(json.dumps(augmented_tokenizer.token_to_index))

Some tokens have special meanings:
- `<PAD>` allows sequences of different lengths to be padded to the same length.
- `<UNK>` is used when an input contains unknown tokens.
- `<START>` occurs at the start of every sequence and is used as the first input when generating a sequence.
- `<STOP>` signifies the end of a sequence (before padding), i.e. that generation should not be continued.

**Demonstrate augmented tokenization**

The following cell converts the SMILES string "CCO" (ethanol) into token indices and then back into a SMILES string.  
As the SMILES are augmented (randomized) prior to tokenization, the results may differ each time.

Also note that the token indices 2 (`<START>`) and 3 (`<STOP>`) are automatically added to the start and end.

In [ ]:
smiles = "CCO"  # ethanol
for _ in range(5):
  tokenized = augmented_tokenizer.smiles_to_token_indexes(smiles)
  print(tokenized.tolist())
  print(augmented_tokenizer.token_indexes_to_smiles(tokenized))

**Demonstrate canonicalized tokenization**

The following cell converts different SMILES strings for ethanol into token indices.  
Note that the SMILES are _canonicalized_ before tokenization, such that the result is the same for all SMILES corresponding to the same molecule.

In [ ]:
def canonical_tokenization_demo(smiles):
  tokenized = canonical_tokenizer.smiles_to_token_indexes(smiles)
  print(tokenized.tolist())
  print(canonical_tokenizer.token_indexes_to_smiles(tokenized))
  print()


canonical_tokenization_demo("CCO")
canonical_tokenization_demo("OCC")
canonical_tokenization_demo("C(O)C")
canonical_tokenization_demo("C(C)O")

## Define model architecture

In [ ]:
class PredictionNetwork(nn.Module):
  def __init__(
    self,
    latent_dim: int,
    n_properties: int,
    n_hidden_layers: int = 3,
    hidden_layer_size: Optional[int] = None,
  ):
    super().__init__()

    if hidden_layer_size is None:
      hidden_layer_size = 4 * latent_dim

    self.layers = nn.Sequential(
      *[
        nn.Sequential(
          nn.Linear(
            hidden_layer_size if i else latent_dim,
            hidden_layer_size,
          ),
          nn.LayerNorm(hidden_layer_size),
          nn.ReLU(),
        )
        for i in range(n_hidden_layers)
      ],
      nn.Linear(hidden_layer_size, n_properties),
    )

  def forward(self, z):
    return self.layers(z)

In [ ]:
class Encoder(nn.Module):
  def __init__(
    self,
    latent_dim: int,
    vocab_size: int,
    pad_idx: int = 0,
    embedding_dim: int = 256,
    num_layers: int = 4,
  ):
    super().__init__()

    self.pad_idx = pad_idx

    # Maps token indices into learned vectors of size embedding_dim.
    # pad_idx will always be mapped to a zero-vector.
    self.embedding = nn.Embedding(
      vocab_size,
      embedding_dim,
      pad_idx,
    )

    # Positional encoding in the form of sinusoidal curves
    # with a different frequency for each dimension
    self.pos_encoding = PositionalEncoding1D(embedding_dim)

    # A transformer block consisting num_layers layers
    # with multi-headed self-attention blocks
    # and feed-forward networks
    self.transformer = nn.TransformerEncoder(
      nn.TransformerEncoderLayer(
        batch_first=True,
        d_model=embedding_dim,
        nhead=4,
      ),
      num_layers,
    )

    # Affine transformations for converting the final embeddings
    # to mu and logvar
    self.mu_layer = nn.Linear(embedding_dim, latent_dim)
    self.logvar_layer = nn.Linear(embedding_dim, latent_dim)

  def forward(self, sequences: torch.Tensor):
    # sequences should have size (batch_size, max_sequence_length)
    # and contain token indices no higher than vocab_size-1

    pad_mask = sequences == self.pad_idx  # mask of padding tokens

    # Map token indices to fixed-size vectors
    sequences = self.embedding(sequences)

    # Add positional information
    sequences = sequences + self.pos_encoding(sequences)

    # Pass embeddings through transformer,
    # while ignoring padding tokens
    sequences = self.transformer(
      sequences,
      src_key_padding_mask=pad_mask,
    )

    # Use the first token of each sequence as a summary
    summary = sequences[:, 0]

    # Project the summary embedding to get the parameters
    # of a normal distribution.
    # We use the logarithm of the variance instead of the
    # standard deviation for better stability.
    mu = self.mu_layer(summary)
    logvar = self.logvar_layer(summary)

    return Normal(mu, torch.exp(0.5 * logvar))

In [ ]:
class Decoder(nn.Module):
  def __init__(
    self,
    latent_dim: int,
    vocab_size: int,
    pad_idx: int = 0,
    start_idx: int = 2,
    end_idx: int = 3,
    embedding_dim: int = 256,
    num_layers: int = 4,
    max_len: int = 500,
  ):
    super().__init__()

    self.pad_idx = pad_idx
    self.start_idx = start_idx
    self.end_idx = end_idx
    self.max_len = max_len

    # Project the latent vector to the embedding space
    self.z_projection = nn.Linear(latent_dim, embedding_dim)

    # Maps token indices into learned vectors of size embedding_dim.
    # pad_idx will always be mapped to a zero-vector.
    self.embedding = nn.Embedding(
      vocab_size,
      embedding_dim,
      pad_idx,
    )

    # Positional encoding in the form of sinusoidal curves
    # with a different frequency for each dimension
    self.pos_encoding = PositionalEncoding1D(embedding_dim)

    # A transformer block consisting num_layers layers
    # consisting of masked multi-headed self-attention blocks,
    # multi-headed cross-attention blocks,
    # and feed-forward networks
    self.decoder = nn.TransformerDecoder(
      nn.TransformerDecoderLayer(
        batch_first=True,
        d_model=embedding_dim,
        nhead=4,
      ),
      num_layers,
    )

    # Project final embeddings to have one output per token
    self.token_layer = nn.Linear(embedding_dim, vocab_size)

  def forward(self, z: torch.Tensor, tokens: torch.Tensor):
    if z.size(0) != tokens.size(0):
      raise ValueError("Arguments must have same batch size")

    # (batch_size, latent_dim) -> (batch_size, 1, embedding_dim)
    z = self.z_projection(z).unsqueeze(1)

    # Map token indices to embedding space
    tokens = self.embedding(tokens)
    
    # Add positional information
    tokens = tokens + self.pos_encoding(tokens)

    # Pass the token embeddings into the transformer,
    # with z as the memory input (conditioning)
    # and a causal attention mask
    transformed = self.decoder(
      tgt=tokens,
      memory=z,
      tgt_mask=nn.Transformer.generate_square_subsequent_mask(
        tokens.size(1), device=tokens.device
      ),
      tgt_is_causal=True,
    )

    # Project embeddings to have one output per token in the vocabulary
    logits = self.token_layer(transformed)

    return logits

  @torch.no_grad()
  def greedy_decode(self, z: torch.Tensor):
    tokens = torch.full(
      (z.size(0), 1),
      self.start_idx,
      device=z.device,
    )
    for i in range(self.max_len):
      # Compute logits for last sequence position
      logits = self(z, tokens)[:, -1]

      # greedy decoding: use index of the maximum value
      greedy = logits.argmax(-1, keepdims=True)

      # Append new token indices to old ones
      tokens = torch.cat([tokens, greedy], dim=1)

      # Stop generating when all sequences contain an end token
      end_mask = (tokens == self.end_idx).any(dim=1)
      if end_mask.all():
        break

    # discard everything after the first end token in each sequence
    end_mask = tokens == self.end_idx
    end_positions = end_mask.int().argmax(dim=1)
    end_mask = end_mask.any(dim=1)
    tokens[end_mask] = tokens[end_mask, : end_positions + 1]

    return tokens

In [ ]:
class VAE(nn.Module):
  def __init__(
    self,
    latent_dim: int,
    vocab_size: int,
    n_properties: int,
    pad_idx: int = 0,
    start_idx: int = 2,
    end_idx: int = 3,
  ):
    super().__init__()

    self.latent_dim = latent_dim

    self.encoder = Encoder(
      latent_dim,
      vocab_size,
      pad_idx,
    )
    self.decoder = Decoder(
      latent_dim,
      vocab_size,
      pad_idx,
      start_idx,
      end_idx,
    )
    self.property_prediction_network = PredictionNetwork(
      latent_dim, n_properties
    )

    self.prior = Normal(0, 1)  # Prior distribution

  def forward(
    self,
    input_seqs: torch.Tensor,
    target_seqs: torch.Tensor,
  ):
    # Pass input sequences to the encoder to obtain distributions
    dists = self.encoder(input_seqs)

    # Sample latent vectors from the encoder distributions
    z = dists.rsample()

    # Pass latent vectors (and teacher-forcing tokens) to the decoder
    logits = self.decoder(z, target_seqs[:, :-1])

    # Predict properties
    pred_props = self.property_prediction_network(z)

    return logits, pred_props, kl_divergence(dists, self.prior)

  def sample(self, n_samples):
    return self.prior.sample((n_samples, self.latent_dim))

In [ ]:
model = VAE(
  latent_dim=64,
  vocab_size=canonical_tokenizer.number_of_tokens,
  n_properties=len(df_train.columns) - 1,
).to(device)

## Training the model

In [ ]:
# create checkpoints/
! [[ -d checkpoints ]] || mkdir checkpoints

In [ ]:
optimizer = torch.optim.Adam(
  model.parameters(),
  lr=1e-3,
)
pred_loss_fn = nn.MSELoss(reduction="sum")
rec_loss_fn = nn.CrossEntropyLoss(
  reduction="sum",
  ignore_index=0,  # ignore padding
)

In [ ]:
n_epochs = 10
batch_size = 64
tb_writer = SummaryWriter()

In [ ]:
def batched(df, batch_size, shuffle=False):
  if shuffle:
    df = df.sample(frac=1)
  return np.array_split(
    df,
    math.ceil(len(df) / batch_size),
  )


def preprocess_batch(batch):
  # tokenize smiles
  input_tokens = pad_sequence(
    [
      augmented_tokenizer.smiles_to_token_indexes(smiles)
      for smiles in batch["smiles"]
    ],
    batch_first=True,
    padding_value=0,
  ).to(device, torch.long)
  target_tokens = pad_sequence(
    [
      canonical_tokenizer.smiles_to_token_indexes(smiles)
      for smiles in batch["smiles"]
    ],
    batch_first=True,
    padding_value=0,
  ).to(device, torch.long)

  # convert properties from dataframe to tensor
  true_props = torch.tensor(
    batch.iloc[:, 1:].to_numpy(),
    device=device,
    dtype=torch.float,
  )

  return input_tokens, target_tokens, true_props


def beta(epoch):
  # linearly anneal beta from 0 to 1 over 50 epochs
  return min(1, epoch / 50)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

In [ ]:
for epoch in range(n_epochs):
  model.train()  # set model to training mode
  batches = batched(df_train, batch_size, shuffle=True)
  for i, b in enumerate(
    tqdm(
      batches,
      desc=f"Epoch {epoch} (training)",
      leave=False,
    )
  ):
    optimizer.zero_grad()  # zero-out gradients
    input_tokens, target_tokens, true_props = preprocess_batch(b)

    logits, pred_props, kl_div = model(input_tokens, target_tokens)
    rec_loss = rec_loss_fn(
      logits[:, : target_tokens.size(1) - 1, :].permute(0, 2, 1),
      target_tokens[:, 1:],
    )
    kl_div = kl_div.sum()
    pred_loss = pred_loss_fn(pred_props, true_props)
    loss = rec_loss + beta(epoch) * kl_div + pred_loss
    loss.backward()  # back-propagate loss
    optimizer.step()  # update model parameters

    # log training loss every 20 steps
    if not (i % 20):
      tb_writer.add_scalars(
        "Loss/Train",
        {
          "Reconstruction": rec_loss,
          "Prediction": pred_loss,
          "KL Divergence": kl_div,
        },
        i + epoch * len(batches),
      )

  # save model checkpoint
  torch.save(
    {
      "model_state_dict": model.state_dict(),
      "optimizer_state_dict": optimizer.state_dict(),
    },
    f"checkpoints/epoch_{epoch:0>4d}.pt",
  )

  model.eval()  # set model to eval mode
  y_true = []
  y_pred = []
  rec_loss = kl_div = pred_loss = 0
  for b in tqdm(
    batched(df_valid, batch_size),
    desc=f"Epoch {epoch} (validation)",
    leave=False,
  ):
    input_tokens, target_tokens, true_props = preprocess_batch(b)
    with torch.no_grad():
      logits, pred_props, kl_div = model(input_tokens, target_tokens)
    rec_loss += rec_loss_fn(
      logits[:, : target_tokens.size(1) - 1, :].permute(0, 2, 1),
      target_tokens[:, 1:],
    ).item()
    kl_div += kl_div.sum().item()
    pred_loss += pred_loss_fn(pred_props, true_props).item()
    y_true.append(true_props)
    y_pred.append(pred_props)

  r2 = r2_score(y_true, y_pred)
  tb_writer.add_scalars(
    "R2/Validation",
    r2,
    epoch,
  )
  print(
    f"Epoch {epoch}: \tReconstruction loss: {rec_loss:.2f} \t R²: {r2:.4f}"
  )

## Load parameters from checkpoint

In [ ]:
checkpoint = torch.load("checkpoints/epoch_0009.pt", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

## Optimization in latent space

In [ ]:
# function that should be minimized
def scoring_function(pred_props: torch.Tensor):
  # mu / D, homo / Eh, lumo / Eh, gap / Eh
  target_props = [[0, 0, 0, 1.2]]
  target_weights = [[0, 0, 0, 1]]

  # apply normalization
  target_props = scaler.transform(target_props)

  # convert to tensor of same shape as pred_props
  target_props = torch.tensor(
    target_props,
    device=device,
  ).expand(pred_props.size())
  target_weights = torch.tensor(
    target_weights,
    device=device,
  ).expand(pred_props.size())

  # weighted mean-squared-error
  score = (((pred_props - target_props) ** 2) * target_weights).mean(
    dim=-1
  )
  return score

In [ ]:
def cross_over(population):
  # population: (N, latent dim)
  N = population.size(0)

  i, j = torch.triu_indices(N, N, device=device)

  # new population including all pairwise interpolations
  population = (
    population.unsqueeze(0).expand(N, -1, -1)
    + population.unsqueeze(1).expand(-1, N, -1)
  )[i, j, :] / 2

  return population


def mutate(population, sigma=0.5):
  return population + Normal(0, sigma).sample(population.size())

In [ ]:
population = model.sample(1000)
n_generations = 10
k = 20

for i in range(n_generations):
  if i:
    population = cross_over(population)
    population = mutate(population)
  with torch.no_grad():
    pred_props = model.property_prediction_network(population)
  scores = scoring_function(pred_props)

  # sort according to scores
  scores, idx = scores.sort()
  population = population[idx]

  # keep k best results
  scores = scores[:k]
  population = population[:k]

  print(f"Generation {i}: {scores.mean()}")

### Show best candidate molecule

In [ ]:
candidate_latent = population[0].unsqueeze(0)

with torch.no_grad():
  # greedily decode latent vector
  greedy = model.decoder.greedy_decode(candidate_latent).squeeze(0)

  # predict properties
  mu, homo, lumo, gap = scaler.inverse_transform(
    model.property_prediction_network(candidate_latent)
  )[0]

print("PREDICTED PROPERTIES")
print(f"Dipole moment: {mu:.3f} D")
print(f"HOMO energy: {homo:.6f} Hartree")
print(f"LUMO energy: {lumo:.6f} Hartree")
print(f"HOMO-LUMO gap: {gap:.6f} Hartree")

smiles = canonical_tokenizer.token_indexes_to_smiles(greedy)
mol = Chem.MolFromSmiles(smiles)
if mol:
  display(mol)
else:
  print("Invalid SMILES")

### Verify the properties using xtb / pyscf

In order to accelerate the calculation, we use GFN2-xTB for the geometry optimization rather
and B3LYP/6-31G(2df,p) (which was used for the QM9 dataset) only for the final single-point calculation.  
Therefore, the results may not match the values in the dataset exactly.

In [ ]:
smiles = "c1ccccc1O"

In [ ]:
! [[ -d xtb_opt ]] && rm -rf xtb_opt
! mkdir xtb_opt

# Create xyz file
mol = Chem.MolFromSmiles(smiles)
mol = Chem.AddHs(mol)
AllChem.EmbedMolecule(mol)
with open("xtb_opt/mol.xyz", "w") as f:
  f.write(AllChem.MolToXYZBlock(mol, 0))

# Optimize geometry with xtb
! cd xtb_opt && ../data/xtb mol.xyz --opt

In [ ]:
with open("xtb_opt/xtbopt.xyz", "r") as f:
    lines = f.readlines()
xyz_coords = "".join(lines[2:])

mol = gto.Mole()
mol.atom = xyz_coords
mol.basis = '6-31g(2df,p)'
mol.charge = 0
mol.spin = 0
mol.build()

mf = dft.RKS(mol)
mf.xc = 'b3lyp'
mf.kernel()

mo_energies = mf.mo_energy
homo_index = np.where(mf.mo_occ > 0)[0][-1]
lumo_index = homo_index + 1
homo_energy = mo_energies[homo_index]
lumo_energy = mo_energies[lumo_index]

hartree_to_ev = 27.2114
gap_hartree = lumo_energy - homo_energy
dipole = np.linalg.norm(mf.dip_moment(unit="Debye"))

print(f"HOMO energy: {homo_energy:.6f} Hartree")
print(f"LUMO energy: {lumo_energy:.6f} Hartree")
print(f"HOMO-LUMO gap: {gap_hartree:.6f} Hartree")
print(f"Dipole moment: {dipole:.3f} D")